In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import sys

# ------------------ GLOBAL VARIABLES ------------------
nodes_explored = 0
nodes_pruned = 0

# ------------------------------------------------------
# Terminal state check
# ------------------------------------------------------
def is_terminal(state):
    return all(h == 0 for h in state)

# ------------------------------------------------------
# Generate all possible moves
# ------------------------------------------------------
def get_moves(state):
    moves = []
    for i, heap in enumerate(state):
        if heap > 0:
            for remove in range(1, heap + 1):
                new_state = list(state)
                new_state[i] -= remove
                moves.append((state, tuple(new_state)))
    return moves

# ------------------------------------------------------
# Alpha-Beta Pruning with Visualization
# ------------------------------------------------------
def alphabeta(state, alpha, beta, maximizing_player=True, graph=None, parent=None, best_path=None):
    global nodes_explored, nodes_pruned
    nodes_explored += 1

    if graph is None:
        graph = nx.DiGraph()
    if best_path is None:
        best_path = {}

    node_label = str(state)
    if parent is not None:
        graph.add_edge(str(parent), node_label)

    # Terminal State
    if is_terminal(state):
        # CORRECTED: If it's terminal and maximizing_player's turn, 
        # it means minimizing player made the last move and won
        value = -1 if maximizing_player else 1
        print(f"Terminal state {state} → {value}")
        graph.nodes[node_label]["label"] = f"{state}\nVal={value}"
        return value, graph, best_path

    # Generate all possible moves (no sorting to match expected output)
    moves = get_moves(state)

    if maximizing_player:
        print(f"MAX explores {state}")
        value = -sys.maxsize
        best_child = None
        
        for i, move in enumerate(moves):
            print(f"Considering move: {move[0]} → {move[1]}")
            
            # Check for pruning BEFORE exploring the child
            if beta <= alpha:
                remaining_moves = len(moves) - i
                nodes_pruned += remaining_moves
                print(f"[Pruned {remaining_moves} branches at {state} because beta <= alpha]")
                break
            
            child_val, graph, best_path = alphabeta(move[1], alpha, beta, False, graph, state, best_path)
            
            if child_val > value:
                value = child_val
                best_child = move[1]
            
            alpha = max(alpha, value)
        
        graph.nodes[node_label]["label"] = f"{state}\nMAX={value}"
        if best_child:
            best_path[state] = best_child
        return value, graph, best_path
    
    else:  # minimizing player
        print(f"MIN explores {state}")
        value = sys.maxsize
        best_child = None
        
        for i, move in enumerate(moves):
            print(f"Considering move: {move[0]} → {move[1]}")
            
            # Check for pruning BEFORE exploring the child
            if beta <= alpha:
                remaining_moves = len(moves) - i
                nodes_pruned += remaining_moves
                print(f"[Pruned {remaining_moves} branches at {state} because beta <= alpha]")
                break
            
            child_val, graph, best_path = alphabeta(move[1], alpha, beta, True, graph, state, best_path)
            
            if child_val < value:
                value = child_val
                best_child = move[1]
            
            beta = min(beta, value)
        
        graph.nodes[node_label]["label"] = f"{state}\nMIN={value}"
        if best_child:
            best_path[state] = best_child
        return value, graph, best_path

# ------------------------------------------------------
# Basic minimax for comparison
# ------------------------------------------------------
def minimax_basic(state, maximizing_player=True):
    """Basic minimax to count total nodes for comparison."""
    nodes_count = 1
    
    if is_terminal(state):
        return (-1 if maximizing_player else 1), nodes_count
    
    if maximizing_player:
        value = -sys.maxsize
        for move in get_moves(state):
            child_val, child_nodes = minimax_basic(move[1], False)
            nodes_count += child_nodes
            value = max(value, child_val)
        return value, nodes_count
    else:
        value = sys.maxsize
        for move in get_moves(state):
            child_val, child_nodes = minimax_basic(move[1], True)
            nodes_count += child_nodes
            value = min(value, child_val)
        return value, nodes_count

# ------------------------------------------------------
# Draw the game tree with best path highlighted
# ------------------------------------------------------
def draw_tree(graph, best_path, root):
    pos = nx.spring_layout(graph, seed=42)
    labels = nx.get_node_attributes(graph, "label")

    # Highlight the best path edges in red
    path_edges = []
    current = root
    while current in best_path:
        nxt = best_path[current]
        path_edges.append((str(current), str(nxt)))
        current = nxt

    plt.figure(figsize=(14, 10))
    nx.draw(graph, pos, with_labels=False, node_size=2200, node_color="lightblue", edge_color="gray")
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)
    nx.draw_networkx_edges(graph, pos, edgelist=path_edges, edge_color="red", width=2)
    plt.title("Nim Game Tree with Alpha-Beta Pruning (Best Path Highlighted)")
    plt.show()

# ------------------------------------------------------
# Main Function
# ------------------------------------------------------
def play_nim():
    global nodes_explored, nodes_pruned
    
    # Reset counters
    nodes_explored = 0
    nodes_pruned = 0

    # You can modify this to take user input or use a fixed initial state
    # For assignment demonstration, using the example state (3, 4, 5)
    initial_state = (3, 4, 5)
    print(f"Initial State: {initial_state}")
    print("\nRunning Alpha-Beta Pruning...\n")

    # Run alpha-beta pruning
    score, graph, best_path = alphabeta(initial_state, -sys.maxsize, sys.maxsize, True)

    print("\n================= RESULTS =================")
    print(f"Initial State: {initial_state}")
    if initial_state in best_path:
        print(f"Best Move for MAX: {initial_state} → {best_path[initial_state]}")
    else:
        print("No moves possible (Game Over).")

    if score == 1:
        print("Outcome: Winning position")
    else:
        print("Outcome: Losing position")

    print(f"Nodes Explored: {nodes_explored}")
    print(f"Nodes Pruned: {nodes_pruned}")
    
    # Compare with basic minimax
    print("\n=============== COMPARISON ================")
    minimax_val, minimax_nodes = minimax_basic(initial_state, True)
    print(f"Basic Minimax - Nodes explored: {minimax_nodes}")
    print(f"Alpha-Beta - Nodes explored: {nodes_explored}, Nodes pruned: {nodes_pruned}")
    efficiency = ((minimax_nodes - nodes_explored) / minimax_nodes * 100)
    print(f"Efficiency improvement: {efficiency:.1f}% fewer nodes explored")
    print("=========================================\n")

    # Draw game tree
    draw_tree(graph, best_path, initial_state)

# ------------------ ALTERNATIVE: Take user input ------------------
def play_nim_interactive():
    global nodes_explored, nodes_pruned
    nodes_explored = 0
    nodes_pruned = 0

    # Take user input for initial state
    initial_state = tuple(map(int, input("Enter heap sizes separated by space (e.g. 3 4 5): ").split()))
    print(f"\nInitial State: {initial_state}")
    print("\nRunning Alpha-Beta Pruning...\n")

    score, graph, best_path = alphabeta(initial_state, -sys.maxsize, sys.maxsize, True)

    print("\n================= RESULTS =================")
    print(f"Initial State: {initial_state}")
    if initial_state in best_path:
        print(f"Best Move for MAX: {initial_state} → {best_path[initial_state]}")
    else:
        print("No moves possible (Game Over).")

    if score == 1:
        print("Outcome: Winning Position for MAX")
    else:
        print("Outcome: Losing Position for MAX")

    print(f"Nodes Explored: {nodes_explored}")
    print(f"Nodes Pruned: {nodes_pruned}")
    print("=========================================\n")

    # Draw game tree
    draw_tree(graph, best_path, initial_state)

# ------------------ RUN GAME ------------------
if __name__ == "__main__":
    # Use this for assignment demonstration with fixed state (3,4,5)
    play_nim()
    
    # Or uncomment this for interactive input
    # play_nim_interactive()

Initial State: (3, 4, 5)

Running Alpha-Beta Pruning...

MAX explores (3, 4, 5)
Considering move: (3, 4, 5) → (2, 4, 5)
MIN explores (2, 4, 5)
Considering move: (2, 4, 5) → (1, 4, 5)
MAX explores (1, 4, 5)
Considering move: (1, 4, 5) → (0, 4, 5)
MIN explores (0, 4, 5)
Considering move: (0, 4, 5) → (0, 3, 5)
MAX explores (0, 3, 5)
Considering move: (0, 3, 5) → (0, 2, 5)
MIN explores (0, 2, 5)
Considering move: (0, 2, 5) → (0, 1, 5)
MAX explores (0, 1, 5)
Considering move: (0, 1, 5) → (0, 0, 5)
MIN explores (0, 0, 5)
Considering move: (0, 0, 5) → (0, 0, 4)
MAX explores (0, 0, 4)
Considering move: (0, 0, 4) → (0, 0, 3)
MIN explores (0, 0, 3)
Considering move: (0, 0, 3) → (0, 0, 2)
MAX explores (0, 0, 2)
Considering move: (0, 0, 2) → (0, 0, 1)
MIN explores (0, 0, 1)
Considering move: (0, 0, 1) → (0, 0, 0)
Terminal state (0, 0, 0) → -1
Considering move: (0, 0, 2) → (0, 0, 0)
Terminal state (0, 0, 0) → 1
Considering move: (0, 0, 3) → (0, 0, 1)
MAX explores (0, 0, 1)
Considering move: (0, 0, 